## Databricks Homework
Since in July 2025 Databricks Community Edition was deprecated and instead of creating separate cluster they are being provided in serverless mode it will be easier for you to work with data - since all the data and tables will be saving not only when cluster as active.

So, no separate activities for cluser creating should be executed - it will be autoattached/started when you will execute any of the cells below.


Please, create table in the default schema using file Sales_December_2019.csv. On the left found Catalog => Add Data => Drop files to upload, or click to browse => Sales_December_2019.csv After file will be uploaded, just need to confirm that table should be uploaded.

 Make sure that the first row is header selected => Create Table. Table will be created with name that you specified (sales_december_2019 by default) You will be able to change the table name later if needed.

PySpark can process SQL queries as a text. In other words you don't need to switch cell language to SQL.
1. Write data from table that you created into the dataframe using PySpark with SQL query. Show data in the dataframe

In [0]:
# Create DataFrame from SQL query
df = spark.sql("SELECT * FROM sales_december_2019")
df.show()

+--------+--------------------+----------------+----------+--------------+--------------------+
|Order ID|             Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|
+--------+--------------------+----------------+----------+--------------+--------------------+
|  295665|  Macbook Pro Laptop|               1|      1700|12/30/19 00:01|136 Church St, Ne...|
|  295666|  LG Washing Machine|               1|     600.0|12/29/19 07:03|562 2nd St, New Y...|
|  295667|USB-C Charging Cable|               1|     11.95|12/12/19 18:21|277 Main St, New ...|
|  295668|    27in FHD Monitor|               1|    149.99|12/22/19 15:13|410 6th St, San F...|
|  295669|USB-C Charging Cable|               1|     11.95|12/18/19 12:38|43 Hill St, Atlan...|
|  295670|AA Batteries (4-p...|               1|      3.84|12/31/19 22:58|200 Jefferson St,...|
|  295671|USB-C Charging Cable|               1|     11.95|12/16/19 15:10|928 12th St, Port...|
|  295672|USB-C Charging Cable|         

Any notebook can be parameterized using dbutils.widgets. Try to add one parameter "Product_name" and select data from dataframe filtered by value from this parameter. 

2. Select data where product = "product_name" from dataframe using PySpark

In [0]:
dbutils.widgets.text("product_name", "")

In [0]:
product_name = dbutils.widgets.get("product_name")
print(product_name)

Macbook Pro Laptop


As well as in SQL, in PySpark you can use aggregate functions. Package pyspark.sql.functions contains all aggregated function from SQL. Try to perform simple aggregation with dataframe. Don't forget, that column types, which you want to calculate, shoud be numerical.  
3. Calculate the sales for each product, including the number of products sold

In [0]:
# Your code here
filtered_df = df.filter(df["Product"] == product_name)
filtered_df.show()


+--------+------------------+----------------+----------+--------------+--------------------+
|Order ID|           Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|
+--------+------------------+----------------+----------+--------------+--------------------+
|  295665|Macbook Pro Laptop|               1|      1700|12/30/19 00:01|136 Church St, Ne...|
|  295712|Macbook Pro Laptop|               1|      1700|12/10/19 20:02|331 Madison St, N...|
|  295717|Macbook Pro Laptop|               1|      1700|12/25/19 09:51|82 10th St, San F...|
|  295871|Macbook Pro Laptop|               1|      1700|12/28/19 11:19|661 Park St, Dall...|
|  295948|Macbook Pro Laptop|               1|      1700|12/17/19 21:08|863 West St, San ...|
|  295963|Macbook Pro Laptop|               1|      1700|12/08/19 10:21|556 11th St, Aust...|
|  296030|Macbook Pro Laptop|               1|      1700|12/24/19 12:31|698 4th St, Portl...|
|  296068|Macbook Pro Laptop|               1|      1700|12/

In the PySpark you can perform dataframe profiling using one of two special commands or simple aggregated functions. Try to find special commands to complete this task or just use aggregated functions. Hint: please, сhange the column data types based on the data in them

4. Show data profiles output for the new dataframe of table sales_december_2019_csv: row count, min and max value for each column

In [0]:
from pyspark.sql.functions import expr, min, max

df_clean = df \
    .withColumn("Order ID", expr("try_cast(`Order ID` as int)")) \
    .withColumn("Quantity Ordered", expr("try_cast(`Quantity Ordered` as int)")) \
    .withColumn("Price Each", expr("try_cast(`Price Each` as double)"))

# remove bad rows (null after cast)
df_clean = df_clean.filter("`Order ID` IS NOT NULL")

print("Row count:")
print(df_clean.count())

df_clean.select(
    min("Order ID").alias("min_order_id"),
    max("Order ID").alias("max_order_id"),
    min("Quantity Ordered").alias("min_qty"),
    max("Quantity Ordered").alias("max_qty"),
    min("Price Each").alias("min_price"),
    max("Price Each").alias("max_price")
).show()

Row count:
24989
+------------+------------+-------+-------+---------+---------+
|min_order_id|max_order_id|min_qty|max_qty|min_price|max_price|
+------------+------------+-------+-------+---------+---------+
|      295665|      319670|      1|      7|     2.99|   1700.0|
+------------+------------+-------+-------+---------+---------+




5. Add new column to the dataframe from previous task with any default value that you want

Temporary views are processed by cluster and always dropped when the session ends (when the cluster turns off).

6. Create temporary view from task 4 dataframe using PySpark and perform any select using SQL

In [0]:
df_clean.createOrReplaceTempView("sales_view")



In [0]:
%sql
SELECT * FROM sales_view
LIMIT 10

Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
295665,Macbook Pro Laptop,1,1700.0,12/30/19 00:01,"136 Church St, New York City, NY 10001"
295666,LG Washing Machine,1,600.0,12/29/19 07:03,"562 2nd St, New York City, NY 10001"
295667,USB-C Charging Cable,1,11.95,12/12/19 18:21,"277 Main St, New York City, NY 10001"
295668,27in FHD Monitor,1,149.99,12/22/19 15:13,"410 6th St, San Francisco, CA 94016"
295669,USB-C Charging Cable,1,11.95,12/18/19 12:38,"43 Hill St, Atlanta, GA 30301"
295670,AA Batteries (4-pack),1,3.84,12/31/19 22:58,"200 Jefferson St, New York City, NY 10001"
295671,USB-C Charging Cable,1,11.95,12/16/19 15:10,"928 12th St, Portland, OR 97035"
295672,USB-C Charging Cable,2,11.95,12/13/19 09:29,"813 Hickory St, Dallas, TX 75001"
295673,Bose SoundSport Headphones,1,99.99,12/15/19 23:26,"718 Wilson St, Dallas, TX 75001"
295674,AAA Batteries (4-pack),4,2.99,12/28/19 11:51,"77 7th St, Dallas, TX 75001"
